# Discover daily files
Inventory the shared RAW Volume, exclude closed months, and select the oldest file not yet present in the selected environment's Silver table.

In [ ]:
from pathlib import Path
import sys

source_root = next((root / "src" for root in (Path.cwd(), *Path.cwd().parents) if (root / "src").is_dir()), None)
if source_root is not None and str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [ ]:
ENVIRONMENT = "dev"
PROCESSING_DATE = ""
SOURCE_URI_OVERRIDE = ""
try:
    dbutils.widgets.text("environment", ENVIRONMENT)
    dbutils.widgets.text("processing_date", PROCESSING_DATE)
    dbutils.widgets.text("source_uri_override", SOURCE_URI_OVERRIDE)
    ENVIRONMENT = dbutils.widgets.get("environment").strip().lower()
    PROCESSING_DATE = dbutils.widgets.get("processing_date").strip()
    SOURCE_URI_OVERRIDE = dbutils.widgets.get("source_uri_override").strip()
except NameError:
    pass
if ENVIRONMENT not in {"dev", "prod"}:
    raise ValueError("environment must be dev or prod")

In [ ]:
from finops_cloud.config import load_config
from finops_cloud.runtime import get_spark
from finops_cloud.storage.discover_daily import inventory_daily_files, select_daily_file

config = load_config(ENVIRONMENT)
spark_session = get_spark(config.profile)
inventory = inventory_daily_files(spark_session, config, dbutils.fs.ls)
inventory_rows = [item.as_dict() for item in inventory]
if inventory_rows:
    display(spark_session.createDataFrame(inventory_rows).orderBy("processing_date"))
else:
    print(f"No Parquet file found under {config.source_volume}/daily")

In [ ]:
selected = select_daily_file(
    inventory,
    processing_date=PROCESSING_DATE,
    source_uri_override=SOURCE_URI_OVERRIDE,
    config=config,
)
selection = {"has_new_file": selected is not None}
if selected is not None:
    selection.update(selected.as_dict())
print(selection)
try:
    dbutils.jobs.taskValues.set(key="has_new_file", value=selected is not None)
    dbutils.jobs.taskValues.set(key="source_uri", value=selected.source_uri if selected else "")
    dbutils.jobs.taskValues.set(key="processing_date", value=selected.processing_date if selected else "")
    dbutils.jobs.taskValues.set(key="billing_month", value=selected.billing_month if selected else "")
except NameError:
    pass